# 05. Country-level indices

Brings the two sides together. Each **country** gets a topic profile from the posts
written in it and one from the NDC of the Party it filed under, and the indices measure how
far apart they are and which way the online conversation leans. The EU's member states are
separate countries sharing one NDC.

The unit matters: it is what the figures plot and what Table 1 regresses, and aggregating
the published per-post results per country reproduces both of the paper's reported numbers
(95 countries above the threshold, 91 of them with an NDC). The Party view is written out
too, as `indices_2022_by_party.csv`, for anything that needs one row per NDC.

    GA_cj   NDC weight on topic j, averaged over the Party's 500-word chunks
    PP_cj   online weight on topic j, the summed BERTScore recall over the Party's
            posts, normalised so the 38 weights sum to one
    PS_cj   mean topic-weighted stance

    ADI     Jensen-Shannon divergence between GA and PP across the six thematic groups
    OSI     max(0, mean_j PS_cj)
    TVR     log(PP_ij / GA_ij), per topic

## Two corrections carried into this notebook

**ADI is over the six groups**, not the 38 topics. That is what the published code
does; the manuscript's Eq. (7) should say so. A 38-topic distribution is sparse enough
that the divergence would be driven by noise.

**The topic-level TVR needs the renumbering.** T1–T38 is a hand reordering of
BERTopic's 0–37, so T1 is BERTopic topic 1, T2 is topic 0, and so on. The 2025
notebook applied an extra swap of T2↔T35 and T6↔T38 on top of that permutation,
commented as fixing an inconsistency; the topic keywords say it made things worse
(overlap score 182 with the permutation alone, 162 with the swap). Only the
topic-level TVR for those four topics is affected — ADI, OSI and the group-level TVR
run through a different path.

## Everything is computed twice

Once on all 590,488 posts and once on the 276,834 that match a rule naming climate
directly. Hand labelling put the taxonomy's precision at 67.3% overall and 82.8% for
that subset (notebook 02), and notebook 03 found the conjunction-only posts carry
*more* topic weight than their share of the sample rather than less — so whether the
noise moves the indices is an open question that this notebook settles.

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # the modules the notebooks import live in src/


In [1]:
import os

import numpy as np
import pandas as pd

import config as cfg
import aggregate as ag
import regress as rg

O = cfg.OUTPUT_DIR
pd.set_option('display.width', 200)
print("output:", O)

output: …/output


## 1. Load the three sides

Posts carry the place; recall and stance carry the measurements. All three are in the
same order, keyed on `tweet_id` — the assertion below is worth keeping, because a
silent misalignment here would be invisible in every number that follows.

In [2]:
posts = pd.read_parquet(os.path.join(O, 'climate_posts_2022.parquet'),
                        columns=['tweet_id', 'country', 'created_at'])
recall = pd.read_parquet(os.path.join(O, 'topic_recall_2022.parquet'))
stance = pd.read_parquet(os.path.join(O, 'topic_stance_2022.parquet'))
# the post's own stance, which OSI is the plain mean of
sscore = pd.read_parquet(os.path.join(O, 'stance_2022.parquet'),
                         columns=['tweet_id', 'stance_score'])
assert (sscore.tweet_id.to_numpy() == stance.tweet_id.to_numpy()).all()
sscore = sscore.stance_score
ga_chunks = pd.read_csv(os.path.join(O, 'GA_topics.csv'), low_memory=False)

assert (posts.tweet_id.to_numpy() == recall.tweet_id.to_numpy()).all()
assert (posts.tweet_id.to_numpy() == stance.tweet_id.to_numpy()).all()

posts = ag.add_party(posts)
ga_party = ag.ndc_by_party(ga_chunks)
ga_country = rg.ga_by_country(ga_party)

print(f"posts   {len(posts):,}   unmapped place names: {posts.country.isna().sum():,}")
print(f"NDC     {len(ga_chunks):,} chunks -> {len(ga_party)} Parties "
      f"-> {ga_country.party.nunique()} countries")
print(f"countries in the corpus: {posts.shp_country.nunique()}")

posts   590,488   unmapped place names: 132
NDC     2,762 chunks -> 165 Parties -> 191 countries
countries in the corpus: 142


## 2. The indices

`country_indices` takes a boolean mask, so the sensitivity runs below are the same
function on a subset rather than a second implementation of the same arithmetic.

In [3]:
strong = pd.read_parquet(os.path.join(O, 'strong_flag.parquet')).strong.to_numpy()

idx_all, meta_all = ag.country_indices(posts.shp_country, recall, stance, ga_country,
                                       stance_score=sscore, label='all posts')
idx_str, meta_str = ag.country_indices(posts.shp_country, recall, stance, ga_country,
                                       mask=strong, stance_score=sscore,
                                       label='names climate directly')

# the Party view as well, for anything that needs one row per NDC
idx_party, meta_party = ag.country_indices(posts.party, recall, stance, ga_party,
                                           stance_score=sscore, label='by Party')

for m in (meta_all, meta_str, meta_party):
    print(f"{m['label']:<26s} posts {m['posts']:>7,}  "
          f"units >{cfg.MIN_TWEETS_PER_PARTY} posts {m['parties_above_threshold']:>3d}  "
          f"with an NDC {m['parties_with_ndc']:>3d}")

idx_all.to_csv(os.path.join(O, 'indices_2022_by_country.csv'))
idx_str.to_csv(os.path.join(O, 'indices_2022_by_country_strong.csv'))
idx_party.to_csv(os.path.join(O, 'indices_2022_by_party.csv'))
idx_all[['n_posts', 'ADI', 'OSI'] + [f'{g}_TVR' for g in ag.GROUPS]].round(4).head(10)

all posts                  posts 584,730  units >100 posts 104  with an NDC 100
names climate directly     posts 272,335  units >100 posts  83  with an NDC  80
by Party                   posts 584,926  units >100 posts  84  with an NDC  80


,n_posts,ADI,OSI,G1_TVR,G2_TVR,G3_TVR,G4_TVR,G5_TVR,G6_TVR
party,,,,,,,,,
Albania,136,0.0885,0.6765,-0.4407,0.1540,-0.6948,0.1092,1.4772,1.4696
Algeria,104,0.0741,0.2788,0.0442,-0.2818,-0.3773,-0.2681,1.6047,1.8969
Argentina,347,0.1321,0.6888,-1.2287,0.4195,0.0922,-0.3349,1.5486,1.5826
Australia,27026,0.2904,0.6002,-1.6624,1.0603,0.8053,0.6722,2.0183,2.1767
Austria,863,0.0805,0.7903,0.0325,-0.0006,1.8032,-0.4136,0.2365,-0.9484
Azerbaijan,154,0.0946,0.8896,-0.9959,-0.2471,0.1621,0.5431,1.3748,1.3572
Bahamas,191,0.0653,0.3979,-0.4106,-0.4724,0.1061,0.1240,1.2815,1.4464
Bahrain,220,0.0575,0.3773,-0.5364,0.2116,-0.3695,-0.6323,1.0526,1.0717
Bangladesh,2390,0.0888,0.8895,-0.8799,-0.3823,0.2588,0.2847,1.3156,1.3500


In [4]:
### How the country count compares with the published one

for name, idx in [('all posts', idx_all), ('strong rules', idx_str)]:
    print(f"  {name:<14s} {len(idx):>3d} countries above the threshold with an NDC")
print()
print("  published, aggregated the same way: 95 countries above the threshold, 91 with an")
print("  NDC -- the two numbers the manuscript reports (notebook 06, section 2).")

  all posts      100 countries above the threshold with an NDC
  strong rules    80 countries above the threshold with an NDC

  published, aggregated the same way: 95 countries above the threshold, 91 with an
  NDC -- the two numbers the manuscript reports (notebook 06, section 2).


## 3. Does the filter noise move the indices?

Conjunction-only posts are 53% of the sample and about 50% precise. If restricting to
the cleaner half leaves the indices in the same place, the noise is not driving
anything and the full sample can stay the main analysis with this as a robustness
check. If it does not, the main analysis has to be reconsidered.

In [5]:
cmp, n_shared = ag.compare(idx_all, idx_str, 'all', 'strong')
print(f"shared Parties: {n_shared}")
cmp.round(4)

shared Parties: 80


,mean_all,mean_strong,pearson,spearman,mean_abs_change
index,,,,,
ADI,0.1036,0.1017,0.9891,0.9728,0.0065
OSI,0.7345,0.7343,0.9469,0.9463,0.0313
G1_TVR,-0.6972,-0.6737,0.9966,0.9930,0.0372
G2_TVR,0.2225,0.1791,0.9955,0.9931,0.0452
G3_TVR,0.2861,0.3832,0.9989,0.9913,0.0979
G4_TVR,-0.1664,-0.2509,0.9762,0.9664,0.1015
G5_TVR,1.0911,1.0514,0.9968,0.9902,0.0515
G6_TVR,0.9728,0.9769,0.9981,0.9890,0.0388


In [6]:
### Where do they differ most?

d = pd.DataFrame({'ADI_all': idx_all.ADI, 'ADI_strong': idx_str.ADI}).dropna()
d['diff'] = d.ADI_strong - d.ADI_all
print("ADI, largest changes")
print(d.reindex(d['diff'].abs().sort_values(ascending=False).index).head(8).round(4).to_string())

o = pd.DataFrame({'OSI_all': idx_all.OSI, 'OSI_strong': idx_str.OSI}).dropna()
o['diff'] = o.OSI_strong - o.OSI_all
print("\nOSI, largest changes")
print(o.reindex(o['diff'].abs().sort_values(ascending=False).index).head(8).round(5).to_string())

ADI, largest changes
                ADI_all  ADI_strong    diff
party                                      
Serbia           0.2325      0.2143 -0.0182
Finland          0.0835      0.0985  0.0150
Czech Republic   0.0853      0.1001  0.0149
Ghana            0.1390      0.1247 -0.0143
Poland           0.0795      0.0937  0.0141
Netherlands      0.0774      0.0902  0.0128
Greece           0.0848      0.0974  0.0127
Spain            0.0841      0.0961  0.0119

OSI, largest changes
                   OSI_all  OSI_strong     diff
party                                          
Saudi Arabia       0.55614     0.69637  0.14023
Republic of Korea  0.65789     0.53659 -0.12131
Ukraine            0.47980     0.59524  0.11544
Bahrain            0.37727     0.27451 -0.10276
Brazil             0.60927     0.52390 -0.08537
Jordan             0.41304     0.33014 -0.08290
Costa Rica         0.80260     0.72593 -0.07668
Serbia             0.83273     0.90741  0.07468


## 4. Against the published run

The published indices came from 338,520 posts read at 140 characters, and the
rebuild reads the full text of 590,488. Rankings should survive if the exercise is
sound; levels will move, because reading the whole post raises recall against every
topic (mean total recall 0.22 for untruncated posts against 0.39 for truncated ones,
notebook 03).

In [7]:
pub_path = os.path.join(cfg.REVISION_DIR, 'work', 'published_by_country_full.csv')
if os.path.exists(pub_path):
    pub = pd.read_csv(pub_path, index_col=0)
    shared = idx_all.index.intersection(pub.index)
    print(f"shared countries: {len(shared)} of {len(pub)} published, {len(idx_all)} rebuilt")
    for col in ['ADI', 'OSI']:
        x, y = pub.loc[shared, col], idx_all.loc[shared, col]
        print(f"  {col}: pearson {x.corr(y):.4f}  spearman {x.corr(y, method='spearman'):.4f}"
              f"   mean {x.mean():.4f} -> {y.mean():.4f}")
else:
    print("run section 2 of notebook 06 first; it writes")
    print(f"  {pub_path}")

shared countries: 89 of 91 published, 100 rebuilt
  ADI: pearson 0.9880  spearman 0.9747   mean 0.1005 -> 0.1013
  OSI: pearson 0.7222  spearman 0.7277   mean 0.0071 -> 0.7303


## 5. Topic-level TVR

Positive means the online conversation gives a topic more weight than the NDCs do,
negative the reverse. This is what Figure 4 plots against OSI.

In [8]:
tbl = pd.read_csv(os.path.join(O, 'topic_table.csv')).set_index('topic')
tvr = idx_all[[f'T{i}_TVR' for i in range(1, 39)]].replace([np.inf, -np.inf], np.nan)

summary = pd.DataFrame({
    'mean': tvr.mean(),
    'sd': tvr.std(),
    'ci_low': tvr.mean() - 1.96 * tvr.sem(),
    'ci_high': tvr.mean() + 1.96 * tvr.sem(),
    'n_countries': tvr.notna().sum(),
})
summary.index = [c.replace('_TVR', '') for c in summary.index]
summary['group'] = tbl.reindex(summary.index)['group']
summary['topic'] = tbl.reindex(summary.index)['sentence'].str.split(' - ').str[0].str[4:]
summary.to_csv(os.path.join(O, 'tvr_by_topic.csv'))

print("Most over-discussed online")
print(summary.sort_values('mean', ascending=False).head(6)[
    ['mean', 'sd', 'group', 'topic']].round(3).to_string())
print("\nMost under-discussed")
print(summary.sort_values('mean').head(6)[
    ['mean', 'sd', 'group', 'topic']].round(3).to_string())

Most over-discussed online
      mean     sd group                         topic
T35  2.745  0.621    G2   Energy regulation standards
T24  2.698  2.147    G5           Public consultation
T23  2.622  1.025    G1     GHG emissions commitments
T38  2.492  0.798    G3         Climate compatibility
T15  2.354  0.855    G6               Climate finance
T31  2.155  2.018    G2         Aviation and shipping

Most under-discussed
      mean     sd group                        topic
T5  -4.539  1.336    G1         Targets and ambition
T2  -3.512  1.261    G2             Renewable energy
T7  -2.939  1.169    G4     Governance and direction
T20 -2.882  1.214    G2   Freight and transportation
T1  -2.793  1.162    G1          Emission accounting
T4  -1.748  0.842    G3         Adaptation processes


In [9]:
### Group-level TVR and OSI with SD and CI, which Referee 2 asked for (R2.7)

rows = []
for g in ag.GROUPS:
    v = idx_all[f'{g}_TVR'].replace([np.inf, -np.inf], np.nan).dropna()
    o = idx_all[f'{g}_OSI'].replace([np.inf, -np.inf], np.nan).dropna()
    rows.append({'group': g, 'name': cfg.GROUP_NAMES[g], 'n_countries': len(v),
                 'TVR_mean': v.mean(), 'TVR_sd': v.std(),
                 'TVR_ci_low': v.mean() - 1.96 * v.sem(),
                 'TVR_ci_high': v.mean() + 1.96 * v.sem(),
                 'OSI_mean': o.mean(), 'OSI_sd': o.std(),
                 'OSI_ci_low': o.mean() - 1.96 * o.sem(),
                 'OSI_ci_high': o.mean() + 1.96 * o.sem()})
grp = pd.DataFrame(rows).set_index('group')
grp.to_csv(os.path.join(O, 'tvr_by_group.csv'))
grp.round(4)

,name,n_countries,TVR_mean,TVR_sd,TVR_ci_low,TVR_ci_high,OSI_mean,OSI_sd,OSI_ci_low,OSI_ci_high
group,,,,,,,,,,
G1,Emission accounting and targets,100,-0.6606,0.5060,-0.7598,-0.5614,0.8236,0.1132,0.8015,0.8458
G2,Sector mitigation,100,0.1973,0.3271,0.1332,0.2614,0.8257,0.1067,0.8048,0.8466
G3,Adaptation,100,0.2862,0.8548,0.1187,0.4538,0.8160,0.1125,0.7940,0.8381
G4,Governance and regulation,100,-0.1590,0.4699,-0.2511,-0.0669,0.8564,0.0907,0.8386,0.8742
G5,Social inclusion,100,1.0736,0.5677,0.9624,1.1849,0.8570,0.0872,0.8399,0.8741
G6,Finance and carbon markets,100,0.9178,1.0118,0.7195,1.1161,0.8186,0.1144,0.7962,0.8410
